## Example 1: Basic phone number and color validation

In [ ]:
from pydantic_extra_types.color import Color
from pydantic_extra_types.phone_numbers import PhoneNumber
from pydantic import BaseModel

class UserProfile(BaseModel):
    name: str
    phone: PhoneNumber
    favorite_color: Color

# Example 1: Basic phone number and color validation
try:
    user = UserProfile(
        name="Alice Johnson",
        phone="+12025550123",  # Valid US phone number
        favorite_color="blue"
    )
    print(f"User: {user.name}")
    print(f"Phone: {user.phone}")
    print(f"Color: {user.favorite_color}")
    print("Example 1 completed successfully")
    
except Exception as e:
    print(f"Validation error: {e}")
    raise AssertionError(
        f"pydantic_extra_types example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Multiple color format validation

In [ ]:
from pydantic_extra_types.color import Color
from pydantic import BaseModel

class ThemeSettings(BaseModel):
    primary_color: Color
    secondary_color: Color
    background_color: Color

# Example 2: Multiple color format validation
try:
    # Test different color formats
    theme = ThemeSettings(
        primary_color="#FF5733",      # Hex format
        secondary_color="rgb(46, 204, 113)",  # RGB format
        background_color="lightgray"   # Named color
    )
    
    print(f"Primary: {theme.primary_color}")
    print(f"Secondary: {theme.secondary_color}")
    print(f"Background: {theme.background_color}")
    print("Example 2 completed successfully")
    
except Exception as e:
    print(f"Color validation error: {e}")
    raise AssertionError(
        f"pydantic_extra_types example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: International phone number validation

In [ ]:
from pydantic_extra_types.phone_numbers import PhoneNumber
from pydantic import BaseModel, ValidationError

class Contact(BaseModel):
    name: str
    primary_phone: PhoneNumber
    secondary_phone: PhoneNumber = None

# Example 3: International phone number validation
contacts = [
    {"name": "US User", "primary_phone": "+12025550123"},
    {"name": "UK User", "primary_phone": "+442079460958"},
    {"name": "French User", "primary_phone": "+33142561234"},
    {"name": "Invalid User", "primary_phone": "invalid-phone"}
]

for contact_data in contacts:
    try:
        contact = Contact(**contact_data)
        print(f"✓ {contact.name}: {contact.primary_phone}")
    except ValidationError as e:
        print(f"✗ {contact_data['name']}: {e.errors()[0]['msg']}")

print("Example 3 completed")

## Example 4: Complex nested validation with phone and color types

In [ ]:
from pydantic_extra_types.color import Color
from pydantic_extra_types.phone_numbers import PhoneNumber
from pydantic import BaseModel, Field
from typing import List

class BusinessCard(BaseModel):
    name: str = Field(..., min_length=2)
    phone: PhoneNumber
    company_colors: List[Color] = Field(..., min_items=1, max_items=3)
    
class Company(BaseModel):
    name: str
    employees: List[BusinessCard]

# Example 4: Complex nested validation with phone and color types
try:
    company = Company(
        name="Tech Solutions Inc",
        employees=[
            BusinessCard(
                name="John Smith",
                phone="+12025550101",
                company_colors=["#1E88E5", "#43A047"]
            ),
            BusinessCard(
                name="Sarah Johnson", 
                phone="+442079460958",
                company_colors=["blue", "green", "#FF9800"]
            )
        ]
    )
    
    print(f"Company: {company.name}")
    for emp in company.employees:
        print(f"  Employee: {emp.name} | {emp.phone}")
        print(f"    Colors: {', '.join(str(c) for c in emp.company_colors)}")
    
    print("Example 4 completed successfully")
    
except Exception as e:
    print(f"Nested validation error: {e}")
    raise AssertionError(
        f"pydantic_extra_types example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: JSON serialization with Color and PhoneNumber types

In [ ]:
from pydantic_extra_types.color import Color
from pydantic_extra_types.phone_numbers import PhoneNumber
from pydantic import BaseModel, ValidationError, Field
from typing import Optional

class UserPreferences(BaseModel):
    user_id: str
    theme_color: Color = Field(default="white")
    contact_phone: Optional[PhoneNumber] = None
    accent_color: Color = Field(default="#007ACC")

# Example 5: JSON serialization with Color and PhoneNumber types
users_data = [
    {
        "user_id": "user_001",
        "theme_color": "hsl(240, 100%, 50%)",
        "contact_phone": "+12125552468",
        "accent_color": "rgba(255, 99, 71, 0.8)"
    },
    {
        "user_id": "user_002", 
        "theme_color": "#2ECC71",
        "accent_color": "gold"
    }
]

import json

# hsl() serializes to a colour name, rgba() to hex with alpha, the phone number to RFC 3966
expected_json = {
    "user_001": {"user_id": "user_001", "theme_color": "blue", "contact_phone": "tel:+1-212-555-2468", "accent_color": "#ff6347cc"},
    "user_002": {"user_id": "user_002", "theme_color": "#2ecc71", "contact_phone": None, "accent_color": "gold"},
}

print("Processing user preferences:")
for user_data in users_data:
    # both records are valid: a ValidationError here fails the cell
    user_prefs = UserPreferences(**user_data)

    # Use model_dump_json for proper serialization
    json_str = user_prefs.model_dump_json()
    assert json.loads(json_str) == expected_json[user_prefs.user_id], json_str
    assert UserPreferences.model_validate_json(json_str) == user_prefs, json_str

    print(f"\nUser {user_prefs.user_id}:")
    print(f"  Theme: {user_prefs.theme_color}")
    print(f"  Phone: {user_prefs.contact_phone}")
    print(f"  Accent: {user_prefs.accent_color}")
    print(f"  JSON serializable: ✓")

print("\nExample 5 completed successfully")